## Imports

In [1]:
import sys
import os
import torch
import torch.utils.data as data
import torch.nn as nn

sys.path.append(os.path.abspath('../../'))

import neuro_fuzzy_toolbox as nft

In [2]:
import numpy as np

In [3]:
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split

# Binary Classification

## Data

In [4]:
from ucimlrepo import fetch_ucirepo

In [5]:
parkinsons = fetch_ucirepo(id=174)

X = parkinsons.data.features
y = parkinsons.data.targets

In [6]:
x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.5, stratify=y)

In [7]:
unique_classes, counts = np.unique(y_train.values, return_counts=True)
print(unique_classes)
print(counts)

[0 1]
[24 73]


In [8]:
unique_classes, counts = np.unique(y_test.values, return_counts=True)
print(unique_classes)
print(counts)

[0 1]
[24 74]


In [9]:
scaler = MinMaxScaler(feature_range=(0, 1))

x_train = scaler.fit_transform(x_train)

x_test = scaler.transform(x_test)

In [10]:
x_train = torch.from_numpy(x_train)
x_test = torch.from_numpy(x_test)
y_train = torch.from_numpy(y_train.values).squeeze()
y_test = torch.from_numpy(y_test.values).squeeze()

In [11]:
train_loader = data.DataLoader(
    data.TensorDataset(
        x_train, 
        y_train), 
    batch_size = 4, 
    shuffle = True)
x_train = train_loader.dataset.tensors[0]
y_train = train_loader.dataset.tensors[1]

## Model & Training

In [12]:
model = nft.h_ANFIS(
    input_size = 22,
    num_mfs = 15,
    outputs = 2,
    rule_reduced = True,
    output_type='multiclass',
    dtype=torch.float64
)

model.init_premises(x_train)

In [13]:
loss_fn = nn.functional.cross_entropy

optimizer = torch.optim.AdamW
params = {'lr': 0.0001, 'weight_decay': 0.001}
#optimizer = torch.optim.Adam
#params = {'lr': 0.005}
#optimizer = torch.optim.SGD
#params = {'lr': 0.001, 'momentum': 0.9}

early_stopping = nft.EarlyStopping(
    patience=30, 
    delta=0.001
)

In [14]:
trainer = nft.Hybrid_learning_algorithm(
    epochs=500,
    loss_function=loss_fn,
    optimizer=optimizer,
    optimizer_params=params,
    validation=0.3,
    early_stopping=early_stopping
)

In [15]:
trainer(model, train_loader, verbose=True)

Epoch:   1/500 - loss: 0.695076 - validation loss: 0.680584
Epoch:   2/500 - loss: 0.694506 - validation loss: 0.677761
Epoch:   3/500 - loss: 0.693656 - validation loss: 0.676826
Epoch:   4/500 - loss: 0.690157 - validation loss: 0.677437
Epoch:   5/500 - loss: 0.483683 - validation loss: 0.562885
Epoch:   6/500 - loss: 0.452704 - validation loss: 0.536924
Epoch:   7/500 - loss: 0.670721 - validation loss: 0.675673
Epoch:   8/500 - loss: 0.500327 - validation loss: 0.560785
Epoch:   9/500 - loss: 0.670802 - validation loss: 0.675816
Epoch:  10/500 - loss: 0.702918 - validation loss: 0.709142
Epoch:  11/500 - loss: 0.705736 - validation loss: 0.710055
Epoch:  12/500 - loss: 0.716471 - validation loss: 0.740574
Epoch:  13/500 - loss: 0.707938 - validation loss: 0.695467
Epoch:  14/500 - loss: 0.674379 - validation loss: 0.679546
Epoch:  15/500 - loss: 0.676243 - validation loss: 0.686118
Epoch:  16/500 - loss: 0.661596 - validation loss: 0.660084
Epoch:  17/500 - loss: 0.668144 - valida

In [16]:
train_measures = nft.get_measures(model, x_train, y_train)
for measure in train_measures:
    print(measure + ':', train_measures[measure])

Accuracy: 0.845360824742268
Precision: 0.8476575028636886
Recall: 0.845360824742268
F1: 0.8464055369748327
Confusion Matrix: [[17  7]
 [ 8 65]]


In [17]:
test_measures = nft.get_measures(model, x_test, y_test)
for measure in test_measures:
    print(measure + ':', test_measures[measure])

Accuracy: 0.7040816326530612
Precision: 0.7428285999714571
Recall: 0.7040816326530612
F1: 0.7178635179933389
Confusion Matrix: [[14 10]
 [19 55]]
